# Chapter 8: Distributed Email Service
*System Design Interview Volume 2*

## TL;DR

A distributed email service at Gmail scale (1 billion users) separates **sending** and **receiving** flows through **message queues** for independent scaling. The storage layer combines a **custom metadata database** (partitioned by user_id), **S3 for attachments**, **Redis for recent email caching**, and an **Elasticsearch or custom search store**. The NoSQL data model uses **denormalization** (separate read/unread tables) for fast folder queries. **Email deliverability** requires dedicated IPs, slow warm-up, SPF/DKIM/DMARC authentication, and feedback loop processing. **WebSocket** servers push new emails in real-time. The system favors **consistency over availability** during failover.

## Requirements

| Type | Requirement | Detail |
|---|---|---|
| Functional | Send/receive emails | Core email functionality via SMTP |
| Functional | Fetch all emails | List emails in folders |
| Functional | Read/unread filter | Filter by read status |
| Functional | Search | Full-text search by subject, sender, body |
| Functional | Anti-spam/virus | Filter malicious content |
| Non-functional | Reliability | Zero data loss |
| Non-functional | Availability | Replicated across data centers |
| Non-functional | Scalability | Handle 1 billion users |
| Non-functional | Extensibility | Support features beyond legacy protocols |

## Estimation: Email Volume and Storage

In [ ]:
# Back-of-envelope for distributed email service
users = 1_000_000_000
emails_sent_per_user_per_day = 10
emails_received_per_user_per_day = 40

# QPS
send_qps = users * emails_sent_per_user_per_day / 86_400
print(f"Send QPS: {send_qps:,.0f}")

# Metadata storage (1 year)
avg_metadata_kb = 50
metadata_per_year_pb = (
    users * emails_received_per_user_per_day * 365 * avg_metadata_kb * 1024
) / (1024**5)
print(f"\nMetadata storage (1 year): {metadata_per_year_pb:.0f} PB")

# Attachment storage (1 year)
pct_with_attachment = 0.20
avg_attachment_kb = 500
attachment_per_year_pb = (
    users * emails_received_per_user_per_day * 365
    * pct_with_attachment * avg_attachment_kb * 1024
) / (1024**5)
print(f"Attachment storage (1 year): {attachment_per_year_pb:,.0f} PB")

total_pb = metadata_per_year_pb + attachment_per_year_pb
print(f"Total storage (1 year): {total_pb:,.0f} PB")
print(f"\nConclusion: distributed database required")

## High-Level Design

```mermaid
graph TB
    subgraph Clients["Clients"]
        WM["Webmail -- Browser"]
    end

    subgraph Servers["Server Layer"]
        LB["Load Balancer"]
        WS["Web Servers -- HTTP API"]
        RTS["Real-Time Servers -- WebSocket"]
    end

    subgraph Queues["Message Queues"]
        OQ["Outgoing Email Queue"]
        IQ["Incoming Email Queue"]
        EQ["Error Queue"]
    end

    subgraph Workers["Workers"]
        OW["SMTP Outgoing Workers"]
        IW["Mail Processing Workers"]
    end

    subgraph StorageLayer["Storage Layer"]
        MDB[("Metadata DB")]
        ATT["Attachment Store -- S3"]
        CACHE["Distributed Cache -- Redis"]
        SEARCH["Search Store -- Elasticsearch"]
    end

    WM -->|HTTPS| LB --> WS
    WM <-->|WebSocket| RTS
    WS --> OQ
    WS --> EQ
    OQ --> OW
    OW -->|SMTP| EXT["External Mail Servers"]
    EXT -->|SMTP| IQ
    IQ --> IW
    IW --> MDB
    IW --> ATT
    IW --> CACHE
    IW --> SEARCH
    IW --> RTS
    WS --> MDB
```

## Deep Dive

### Email Protocols

| Protocol | Direction | Behavior |
|---|---|---|
| **SMTP** | Send | Standard for server-to-server email transfer |
| **POP** | Receive | Downloads and deletes from server (single device) |
| **IMAP** | Receive | Syncs on-demand, keeps on server (multi-device) |
| **HTTPS** | Both | Modern webmail (Gmail, Outlook via ActiveSync) |

**DNS MX records** route emails to the correct mail server. Lower priority number = higher preference. Sending servers try the lowest-priority MX first and fall back to higher ones.

### Sending and Receiving Flows

```mermaid
graph LR
    subgraph SendFlow["Email Sending Flow"]
        S1["1. User composes email"]
        S2["2. Load balancer -- rate limit"]
        S3["3. Web server validates"]
        S4["4. Outgoing queue"]
        S5["5. SMTP worker -- spam/virus check"]
        S6["6. Store in Sent folder"]
        S7["7. Send to recipient server"]
    end

    S1 --> S2 --> S3 --> S4 --> S5 --> S6
    S5 --> S7
```

```mermaid
graph LR
    subgraph RecvFlow["Email Receiving Flow"]
        R1["1. SMTP load balancer"]
        R2["2. Acceptance policy"]
        R3["3. Store large attachments in S3"]
        R4["4. Incoming queue"]
        R5["5. Spam/virus filter"]
        R6["6. Store: DB + cache + S3"]
        R7["7. Push via WebSocket if online"]
        R8["8. Pull via HTTP if offline"]
    end

    R1 --> R2 --> R3 --> R4 --> R5 --> R6 --> R7
    R6 --> R8
```

### Email Data Model (NoSQL)

```mermaid
graph TB
    subgraph DataModel["Key Tables"]
        T1["folders_by_user<br/>---<br/>user_id K<br/>folder_id<br/>folder_name"]
        T2["emails_by_folder<br/>---<br/>user_id K<br/>folder_id K<br/>email_id TIMEUUID Cl DESC<br/>from, subject, preview<br/>is_read"]
        T3["emails_by_user<br/>---<br/>user_id K<br/>email_id TIMEUUID Cl<br/>from, to, subject, body<br/>attachments"]
    end
```

**Key design decisions:**
- **Partition key**: user_id ensures all data for one user lives on one shard
- **Clustering key**: TIMEUUID for chronological ordering (newest first)
- **Denormalization**: separate `read_emails` and `unread_emails` tables for fast filtering

In [ ]:
# Illustrate why denormalization is needed for read/unread
# In a relational DB, we'd just add WHERE is_read = true
# In NoSQL, we can only query on partition + clustering keys

tables = {
    "emails_by_folder": {
        "partition_key": "(user_id, folder_id)",
        "clustering_key": "email_id DESC",
        "can_filter": ["user_id", "folder_id", "email_id"],
        "cannot_filter": ["is_read"],  # Not a key column!
    },
    "read_emails": {
        "partition_key": "(user_id, folder_id)",
        "clustering_key": "email_id DESC",
        "can_filter": ["user_id", "folder_id", "email_id"],
        "note": "Only contains read emails -- no filter needed",
    },
    "unread_emails": {
        "partition_key": "(user_id, folder_id)",
        "clustering_key": "email_id DESC",
        "note": "Only contains unread emails -- no filter needed",
    },
}

for name, schema in tables.items():
    print(f"Table: {name}")
    print(f"  Partition key: {schema.get('partition_key', 'N/A')}")
    print(f"  Clustering key: {schema.get('clustering_key', 'N/A')}")
    if "cannot_filter" in schema:
        print(f"  Cannot filter on: {schema['cannot_filter']}")
    if "note" in schema:
        print(f"  Note: {schema['note']}")
    print()

print("Trade-off: denormalization makes writes more complex")
print("(marking read = delete from unread + insert to read)")
print("but makes read queries fast at scale.")

### Conversation Threading

Threads are reconstructed from email headers:

| Header | Purpose |
|---|---|
| **Message-Id** | Unique ID generated by sending client |
| **In-Reply-To** | Message-Id of the parent email |
| **References** | List of all Message-Ids in the thread chain |

Algorithms like **JWZ threading** reconstruct the conversation tree from these headers.

### Email Search

| Feature | Elasticsearch | Custom Search Engine |
|---|---|---|
| **Scalability** | Scalable to some extent | Optimized for email patterns |
| **System complexity** | Two systems (DB + ES) to maintain | One integrated system |
| **Data consistency** | Hard to keep DB and ES in sync | Single copy of data |
| **Data loss** | Rebuild index from primary store | No separate index to lose |
| **Engineering effort** | Easy to integrate | Significant custom development |

**Write-heavy pattern**: email search has far more index writes (every send/receive/delete) than search reads (only when user searches). **LSM trees** optimize for this by doing only sequential disk writes.

```mermaid
graph TB
    subgraph LSM["LSM Tree for Email Search Index"]
        L0["Level 0 -- In-Memory Cache"]
        L1["Level 1 -- Disk SSTable"]
        L2["Level 2 -- Disk SSTable"]
        L3["Level 3 -- Disk SSTable"]
    end

    NEW["New Email Arrives"] --> L0
    L0 -->|"Flush when full"| L1
    L1 -->|"Merge/compact"| L2
    L2 -->|"Merge/compact"| L3
```

### Email Deliverability

In [ ]:
# Email deliverability checklist
deliverability_factors = [
    ("Dedicated IPs", "Use dedicated IPs for sending (not shared)"),
    ("Email classification", "Separate IPs for marketing vs transactional"),
    ("IP warm-up", "Gradually increase volume over 2-6 weeks"),
    ("Ban spammers", "Quick detection and banning before reputation damage"),
    ("SPF", "Sender Policy Framework -- authorize sending IPs"),
    ("DKIM", "DomainKeys Identified Mail -- cryptographic signature"),
    ("DMARC", "Policy for handling SPF/DKIM failures"),
    ("Feedback loops", "Process hard bounces, soft bounces, and complaints"),
]

print("Email Deliverability Checklist:")
print("=" * 65)
for i, (factor, desc) in enumerate(deliverability_factors, 1):
    print(f"  {i}. {factor:<22} {desc}")

print("\nFeedback types:")
print("  - Hard bounce: invalid recipient address (permanent)")
print("  - Soft bounce: temporary failure (ISP busy)")
print("  - Complaint: user clicked 'Report Spam'")
print("\nEach type gets its own processing queue for separate handling.")

### Scalability and Availability

Data is replicated across **multiple data centers**. Users connect to the nearest one. During network partition, the system **pauses mailbox access** rather than serving stale data (consistency over availability).

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Relational DB | ACID, familiar SQL | Not ideal for large email bodies (KB-MB) |
| Custom NoSQL | Optimized for email patterns, low disk I/O | Huge engineering investment |
| S3 for attachments | Scalable, cheap for large blobs | Extra hop for retrieval |
| Cassandra for attachments | Co-located with metadata | 1MB practical blob limit, cache issues |
| Elasticsearch search | Easy integration, full-text search | Two systems, sync complexity |
| Custom LSM search | Single system, write-optimized | Major engineering effort |
| WebSocket for real-time | Elegant, bidirectional | Browser compatibility issues |
| Long polling fallback | Universal browser support | Higher server load, more latency |
| Consistency over availability | No stale reads | Mailbox paused during failover |
| Availability over consistency | Always accessible | Risk of conflicts/stale data |

## Key Takeaways

1. **Decouple sending and receiving** with message queues for independent scaling
2. **Partition by user_id** -- email access patterns are per-user, making sharding natural
3. **Denormalize for NoSQL** -- separate read/unread tables trade write complexity for fast reads
4. **Attachments belong in S3**, not in the metadata database (25MB vs 1MB blob limits)
5. **Email deliverability is hard**: IP reputation, authentication (SPF/DKIM/DMARC), and feedback loops are critical
6. **Search is write-heavy** in email systems (opposite of web search) -- LSM trees optimize for this
7. **WebSocket** provides real-time push; HTTP polling is the fallback
8. **Favor consistency**: pause mailbox during failover rather than serving stale data

## Related Concepts

- [[email-protocols]] -- SMTP, POP, IMAP, and DNS MX records
- [[distributed-mail-architecture]] -- web servers, real-time servers, and queue-based flows
- [[email-data-model]] -- NoSQL tables with denormalization for read/unread
- [[email-search]] -- Elasticsearch vs custom LSM-based engine
- [[email-deliverability]] -- SPF, DKIM, DMARC, and feedback processing
- [[email-scalability-availability]] -- multi-datacenter replication and consistency trade-offs